### eleconomista webscraper

##### Setup

In [ ]:
import requests
from requests.exceptions import ConnectionError 
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from collections import Counter
import time
import json

In [ ]:
with open("headers.json", "r") as f:
    headers_dict_1 = json.load(f)

In [ ]:
with open("headers_2.json", "r") as f:
    headers_dict_2 = json.load(f)

In [ ]:
csv_entries = pd.read_csv('electrical_try.csv', encoding ='latin1', names = ['company names'])
csv_entries['names_sans_accents'] = csv_entries['company names'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

In [ ]:
def url_search_term(business_name: str):
    business_name = business_name.replace(",", "")
    business_name = business_name.replace(".", "")
    business_name = business_name.replace("(", "")
    business_name = business_name.replace(")", "")
    business_name = business_name.replace(' & ', " ")
    url_st = business_name.upper().replace(' ', '-')
    return url_st

##### Test cells

In [ ]:
later_entries = csv_entries.iloc[1198:,:]
later_entries['company_names_sans_accents'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

In [ ]:
csv_entries.iloc[2700]

In [ ]:
company = csv_entries.iloc[0]['names_sans_accents']

mini_header_dict = {
            'User-Agent' : headers_dict_1['48']
        }

baseurl = 'https://empresite.eleconomista.es/Actividad/'
header_index = 49

search_term = url_search_term(company)
search_url = baseurl + search_term + '/'
print(search_url)
r = requests.get(search_url,headers=mini_header_dict, timeout=30)
print(r.status_code)
soup = BeautifulSoup(r.content)
url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))
url_results[0]['href']

name_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
name_result

In [ ]:
head = {'User-Agent' : headers_dict_2['49']}

r=requests.get(link_list[52],headers=head)
soup = BeautifulSoup(r.content)
soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]

In [ ]:
test =  requests.Session()
test2 = test.get(link_list[0], headers=head)
souped = BeautifulSoup(test2.content)
souped
souped.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]

##### Functions + cells to run them

In [ ]:
def business_search_rotate(csv_entries: pd.DataFrame, start_ind: int, n: float, header_dict: dict):
    """
    This function searches the baseurl site for the business names given in the DataFrame csv_entries.
    
    Parameters
    ----------
    """
    link_list = []
    result_names = []
    baseurl = 'https://empresite.eleconomista.es/Actividad/'
    header_index = start_ind

    for i, company in enumerate(tqdm(list(csv_entries['names_sans_accents']))):
        search_term = url_search_term(company)
        search_url = baseurl + search_term + '/'
        mini_header_dict = {
            'User-Agent' : header_dict[str(header_index)]
        }
        try:
            r = requests.get(search_url,headers=mini_header_dict, timeout=30)
            soup = BeautifulSoup(r.content)
            # print(company)
            # print(str(r.status_code))
            try:
                if str(r.status_code)== '200':
                    # first_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                    # first_result = first_result.capitalize().split(' ')[0]
                    # company_name_start = company.split(' ')[0]
                    
                    # if first_result.split(' ')[0]==company.split(' ')[0]:
                    url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                    name_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                    link_list.append(url_results)
                    result_names.append(name_result)
                elif str(r.status_code)== '429':
                    header_index = header_index + 1
                    try:
                        r = requests.get(search_url,headers=mini_header_dict, timeout=30)
                        soup = BeautifulSoup(r.content)
                        url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                        name_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                        link_list.append(url_results)
                        result_names.append(name_result)
                    except:
                        link_list.append('reset but result issue')
                        result_names.append('reset but result issue')
                else:
                    link_list.append('possible 404 not found')
                    result_names.append('possible 404 not found')
            except IndexError:
                link_list.append('no results found')
                result_names.append('no results found')
            time.sleep(n)
        except:
            link_list.append('requests issue')
            result_names.append('requests issue')

    print(f'The cooldown this time was {n} seconds.')
    print(f'Last index in dict used was {header_index}')
    return link_list, result_names

In [ ]:
link_list, result_names = business_search_rotate(csv_entries, start_ind=0, n = 2, header_dict=headers_dict_1)

In [ ]:
Counter(link_list)

In [ ]:
def cleanup_found_links_rotate(df: pd.DataFrame, link_list: list, result_names: list) -> None:
    """
    This function adds urls to company listings that were found and removes rows of companies which were not found listed.

    Parameters
    ----------
    'df' : pd.DataFrame
        DataFrame of companies from csv import.
    'link_list' : list of links found from 'business_search' 
    """
    df = df.copy()
    df.reset_index(drop=True,inplace=True)
    df['found_links'] = pd.DataFrame(link_list, columns=['found_links'])
    df['result_names'] = pd.DataFrame(result_names, columns=['result_names'])
    excluded_rows = df[(df['found_links']=='possible 404 not found') | (df['found_links']=='reset but result issue') | (df['found_links']=='429 issue') | (df['found_links']=='requests issue')].index
    df.drop(index=excluded_rows, inplace=True)
    df = df.drop_duplicates(subset=['found_links'], keep=False) #mutiplicities of url found are generally because of search issues
    df = df.dropna(subset='found_links')
    df.reset_index(drop=True,inplace=True)

    return df

In [ ]:
entries_infoadded = cleanup_found_links_rotate(csv_entries, link_list=link_list,result_names=result_names)

In [ ]:
def get_contact_info_rotate(df: pd.DataFrame, start_ind: int, n: int, header_dict: dict):
    """
    This company scrapes info of companies for which links were found.

    Parameters
    ----------
    'df' : pd.DataFrame
    """
    phonenumbers_found = []
    urls_found = []
    emails_found = []
    header_index = start_ind

    for i, link in enumerate(tqdm(df['found_links'])):
        mini_header_dict = {
            'User-Agent' : header_dict[str(header_index)]
        }
        try:
            r = requests.get(link,headers=mini_header_dict)
            if r.status_code == 429:
                header_index = header_index + 1
                r = requests.get(link,headers=mini_header_dict, timeout=20)
            if (r.status_code != 200) & (r.status_code != 429):
                print(str(r.status_code))
            soup = BeautifulSoup(r.content)
            try:
                found_url = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline'))[0]['href']
            except:
                found_url='not found'
            urls_found.append(found_url)
            try:
                found_email = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]
            except:
                found_email = 'not found'
            emails_found.append(found_email)
            try:
                found_phone = soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text
            except:
                found_phone = 'not found'
            phonenumbers_found.append(found_phone)
        # except:
        #     print(str(r.status_code))
        #     break
        except ConnectionError:
            found_url='connection error not found'
            urls_found.append(found_url)
            found_email = 'connection error not found'
            emails_found.append(found_email)
            found_phone = 'connection error not found'
            phonenumbers_found.append(found_phone)
        time.sleep(n)

    info_dict = {'phones': phonenumbers_found, 'urls': urls_found, 'emails': emails_found}
    print(f'The cooldown this time was {n} seconds.')
    print(f'Last index in dict used was {header_index}')
    return info_dict

In [ ]:
contact_info = get_contact_info_rotate(entries_infoadded, start_ind=0, n=2, header_dict=headers_dict_2)

In [ ]:
def add_found_info(df: pd.DataFrame, contact_info_dict: dict)-> None:
    """
    compiles phone numbers, emails, urls into df and fixes some formatting issues.

    Parameters
    ----------
    'df' : pd.DataFrame
    'contact_info_dict' : dict
    """
    phone_list = contact_info_dict['phones']
    url_list = contact_info_dict['urls']
    email_list = contact_info_dict['emails']
    
    df['phones'] = pd.DataFrame(phone_list, columns=['phones'])
    df['urls'] = pd.DataFrame(url_list, columns=['urls'])
    df['emails'] = pd.DataFrame(email_list, columns=['emails'])

    # Cleaning Results:
    df['urls'] = df['urls'].apply(lambda x: 'not found' if x.startswith('mailto')==True else x)
    df['urls'] = df['urls'].apply(lambda x: 'not found' if x.startswith('https://ranking-empresas.eleconomista.es/')==True else x)

    # Format corrections:
    df['urls'] = df['urls'].apply(lambda x: x[2:] if x.startswith('//')==True else x)
    df['emails'] = df['emails'].apply(lambda x: x.split(':')[1] if x.startswith('mailto')==True else x)
    # df.drop(columns=['found_links','names_sans_accents'],inplace=True)

    # Removing companies w no info:
    no_info_indices = df.loc[(df['emails']=='not found') & (df['phones']=='not found') & (df['urls']=='not found')].index
    df.drop(index = no_info_indices, inplace=True)
    # print(f'no info found for {len(no_info_indices)} businesses for which there was at least 1 search result.')
    df.reset_index(drop=True, inplace=True)
    return None

In [ ]:
add_found_info(df=entries_infoadded, contact_info_dict=contact_info)

In [ ]:
entries_infoadded.to_csv('food_again_entries_eleconomista.csv')